# Baseline: HistGradientBoosting with feature cleanup

Useful notebook for tabular competitions with mostly numeric data.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import mean_squared_error, f1_score

DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

## Basic cleanup

In [ ]:
TARGET_CANDIDATES = ["target", "label", "y", "price", "days", "occupied_days"]
target_col = next((c for c in TARGET_CANDIDATES if c in train.columns), train.columns[-1])

for df in [train, test]:
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype("string").str.strip()

X = train.drop(columns=[target_col])
y = train[target_col]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

## Ordinal encoding pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Model selection and evaluation

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

is_regression = y.dtype.kind in "fiu" and y.nunique() > 20
model = HistGradientBoostingRegressor(random_state=42) if is_regression else HistGradientBoostingClassifier(random_state=42)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])
pipeline.fit(X_train, y_train)
valid_pred = pipeline.predict(X_valid)

if is_regression:
    print({"rmse": mean_squared_error(y_valid, valid_pred, squared=False)})
else:
    print({"f1_macro": f1_score(y_valid, valid_pred, average="macro")})